In [48]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import openpyxl
import re 

import warnings
warnings.filterwarnings("ignore")

In [49]:
start_time = time.time()

In [50]:
bd.projects.set_current('nitrous_oxide_production_monte_carlo')

In [51]:
sorted(list(bd.databases))

['ecoinvent-3.10-biosphere', 'ecoinvent-3.10-cutoff', 'nitrous-oxide-ei310']

In [52]:
methods = [('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]
len(methods)

1

In [53]:
methods

[('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]

In [54]:
db = bd.Database('nitrous-oxide-ei310')
acts = [act for act in db if 'hydrogen peroxide production' in act['name'] or 'nitrous oxide production' in act['name']]
len(acts)

10

In [55]:
mc_df = pd.DataFrame()

In [56]:
demands = [{act:1} for act in acts]
all_demands = {k: 1 for demand in demands for k in demand}
lca = bc.LCA(demand=all_demands, method=methods[0], use_distributions=True)
lca.lci()

C_matrices = {}
for method in methods:
    lca.switch_method(method)
    C_matrices[method] = lca.characterization_matrix.copy()
    
results = {
    act['name']: [] for act in acts
}

for _ in range(500):
    next(lca)
    for index, demand in enumerate(demands):
        lca.lci({key.id: value for key, value in demand.items()})
        for method in methods:
            name = str(list(demand.keys())[0])
            name = name.replace("dict_keys([", "").replace("])", "").replace("'", "")
            name = name[:name.find(' (')]
            results[name].append(
                (C_matrices[method] * lca.inventory).sum()
            )

for key, value in results.items():
    mc_df[key] = pd.Series(value)

In [57]:
mc_df

,"nitrous oxide production, AON 100, fossil ammonia","nitrous oxide production, AON 90 cryogenic, green ammonia","hydrogen peroxide production, AO, green hydrogen","nitrous oxide production, AON 90 membrane, green ammonia","nitrous oxide production, AON 100, green ammonia","nitrous oxide production, AON 90 cryogenic, fossil ammonia","hydrogen peroxide production, AO, fossil hydrogen","nitrous oxide production, BAU, green ammonia","nitrous oxide production, AON 90 membrane, fossil ammonia","nitrous oxide production, BAU, fossil ammonia"
0,2.858570,2.421137,0.656465,1.178738,1.019460,4.458407,1.315517,1.532894,3.209400,3.542769
1,3.113323,2.251113,0.586396,1.037305,0.892571,4.711147,1.235355,1.367461,3.489360,3.794416
2,2.967337,2.330496,0.608728,1.154355,0.998366,4.511619,1.236383,1.481092,3.328404,3.632886
3,2.853198,2.119043,0.505909,0.857765,0.729948,4.471068,1.349687,1.188223,3.202161,3.508621
4,3.153269,2.497699,0.655644,1.272023,1.103967,4.767809,1.306393,1.618460,3.534771,3.858046
...,...,...,...,...,...,...,...,...,...,...
495,3.354365,2.185548,0.535833,1.016475,0.873461,4.933764,1.438905,1.348814,3.755777,4.060075
496,2.641761,1.993274,0.533858,0.865315,0.736458,4.103869,1.352130,1.206095,2.969064,3.288310
497,2.646402,1.892134,0.423290,0.693055,0.580003,4.181182,1.125447,1.036591,2.974679,3.294859
498,2.350266,2.288248,0.543112,1.041768,0.896201,3.898985,1.337019,1.373176,2.647281,2.962255


In [58]:
mc_df.to_excel(os.path.join('..', 'results', 'monte_carlo.xlsx'))